
1_setup : 

- Create Schema, volume
- Manually upload the History file and the Statement file to the volume.
- Validate the listing.

2_1_Statement_Read(Portfolio and Activity raw tables by reading PDF) : raw_portfolio_statement and raw_statement_activity

- Install the PyPDF2 library and restart the python kernel.
- Read the PDF for Portfolio_Activity section from statement and load into volume as portfolio_extract.csv.
- Read the portfolio_extract.csv, create a dataframe and table as raw_portfolio_statement.
- Read the PDF for Account Activity section from statement and create dataframe and load into volume as Account_activity.csv.
- Read the activity_extract.csv, create a dataframe and table as raw_portfolio_statement.

2_2_Read_Inv_Types : raw_invest_types
      
- Read Types.txt from volume storage and create a table raw_invest_types

2_3_Read_Report_File_Volume: history_table

- Read Invest_Dec.csv file from volume and create a view raw_till_nov_temp.
- create one history_table by using raw_investments(transactions from report) and raw_statement_activity and (transactions from monthly statement).

      Settle_Date
      Settle_Month_Year
      Instrument
      Trans_Code
      Quantity
      Price
      Amount

3_raw_data_transformation: valid_transactions, misc_transactions and current_portfolio_value
- create a valid_transactions table for Buy, Sell and CDIV transactions and misc_transactions for all other transactions.
- Apply the required transformation for valid trasaction for the amount field.
- create current_portfolio_value using PDF portfolio values received from raw_portfolio_statement 

4_Silver_layer:  monthly_inv_details and current_details

Building the current_details as of statement 

      Instrument
      Total Qty
      Current Price
      Current Market Value
      Total Buy
      Total Sell
      Total CDIV
      Statement Date
      Sector
      Industry type
      Investment Category
      Present Status
      Current Date
       


      

    



In [0]:
%python
%pip install PyPDF2
dbutils.library.restartPython()

In [0]:
%python
import re
import csv
from PyPDF2 import PdfReader

def extract_portfolio_margin(pdf_path):
    reader = PdfReader("/Volumes/workspace/investment_vision/files_volume/Account statement - 2025-12-31.pdf")

    # Extract all text
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"

    # Locate Portfolio Summary section
    start = full_text.find("Portfolio Summary")
    if start == -1:
        raise ValueError("Portfolio Summary section not found")

    # End when Account Activity begins
    end = full_text.find("Account Activity", start)
    if end == -1:
        end = len(full_text)

    section = full_text[start:end]

    # Regex for margin rows
    pattern = r"([A-Z]{2,5})\s+Margin\s+([\d\.]+)\s+\$([\d\.]+)\s+\$([\d,\.]+)"
    matches = re.findall(pattern, section)

    return matches


# Example usage:
pdf_file = "Account statement - 2025-12-31.pdf"
portfolio_margin = extract_portfolio_margin(pdf_file)

for row in portfolio_margin:
    print(row)

# Save CSV
csv_path = "/Volumes/workspace/investment_vision/files_volume/portfolio_extract.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ticker", "qty", "price", "market_value"])
    writer.writerows(portfolio_margin)

print("CSV created:", csv_path)


In [0]:
%skip
drop table investment_vision.raw_portfolio_statement

In [0]:
%python
from pyspark.sql.functions import col

raw_portfolio_df = spark.read.csv(
    "/Volumes/workspace/investment_vision/files_volume/portfolio_extract.csv",
    header=True
)

raw_portfolio_df = (
    raw_portfolio_df
    .withColumn("qty", col("qty").cast("string"))
    .withColumn("price", col("price").cast("string"))
    .withColumn("market_value", col("market_value").cast("string"))
)

display(raw_portfolio_df)

raw_portfolio_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.investment_vision.raw_portfolio_statement"
)

In [0]:
%python
import re
from PyPDF2 import PdfReader

def extract_account_activity_margin(pdf_path):
    reader = PdfReader("/Volumes/workspace/investment_vision/files_volume/Account statement - 2025-12-31.pdf")

    # Extract all text
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"

    # 1) Find start of Account Activity
    start = full_text.find("Account Activity")
    if start == -1:
        raise ValueError("Account Activity section not found")

    # 2) Find end using the TRUE final marker
    end_marker = "Total Funds Paid and Received"
    end = full_text.find(end_marker, start)
    if end == -1:
        end = len(full_text)

    activity_text = full_text[start:end]

    # 3) Normalize: collapse multi-line rows
    # Remove description/CUSIP lines
    cleaned = []
    for line in activity_text.split("\n"):
        if "Margin" in line:
            cleaned.append(line.strip())

    normalized = " ".join(cleaned)

    # 4) Regex for Margin transactions
    pattern = (
        r"([A-Z]{2,5})\s+Margin\s+"
        r"(Buy|Sell|CDIV|SLIP|DTAX|XENT_CC|INT|DCF)\s+"
        # r"(\d/\d/\d)\s*"
        r"(\d{2}/\d{2}/\d{4})\s*"
        r"([\d\.]*)\s*"
        r"\$?([\d\.]*)\s*"
        r"\$?([\d\.]*)\s*"
    )

    matches = re.findall(pattern, normalized)
    return matches


# Example usage
pdf_file = "Account statement - 2025-12-31.pdf"
activity_margin = extract_account_activity_margin(pdf_file)

print(len(activity_margin))
for row in activity_margin:
    print(row)

# Save CSV
csv_path = "/Volumes/workspace/investment_vision/files_volume/account_activity.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ticker", "transaction_type","date","qty", "current_price", "amount"])
    writer.writerows(activity_margin)

print("CSV created:", csv_path)

In [0]:
%skip
%python
import csv

output_path = "/Volumes/workspace/investment_vision/files_volume/account_activity.csv"

with open(output_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["symbol","transaction_type","date","qty", "current_price", "amount"])
    for row in activity_margin:
        writer.writerow([
            row[0],
            row[1],
            row[2],
            row[3],
            row[4],
            row[5]
        ])

In [0]:
%python
raw_portfolio_df = spark.read.csv("/Volumes/workspace/investment_vision/files_volume/account_activity.csv", header=True)
raw_portfolio_df.display()

raw_portfolio_df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable(
    'workspace.investment_vision.raw_statement_activity')